In [3]:
import pandas as pd

# === INPUT FILES (update paths if needed) ===
mapping_file = "D:/Tushar/main_with_subs_only.xlsx"
indent_file  = "D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"

# === LOAD DATA ===
df_mapping = pd.read_excel(mapping_file)
df_indent   = pd.read_excel(indent_file)

# === RENAME COLUMNS ===
df_mapping = df_mapping.rename(columns={
    'Main_Label': 'Child_Part',
    'Sub_Label':  'Switch_Part',
    'Main_Count': 'Qty_per_Switch',
    'Sub_Count':  'Historical_Total'  # not used in this calculation
})

df_indent = df_indent.rename(columns={'Part number': 'Switch_Part'})

# === MONTH COLUMNS ===
month_cols = ["Feb'26", "Mar'26", "Apr'26", "May'26", "Jun'26", "Jul'26"]

# Create clean month names once (Feb'26 → Feb26)
clean_months = [m.replace("'", "") for m in month_cols]

# === MERGE ===
df_merged = pd.merge(
    df_mapping[['Child_Part', 'Switch_Part', 'Qty_per_Switch']],
    df_indent[['Switch_Part'] + month_cols],
    on='Switch_Part',
    how='left'
)

# === CALCULATE DAILY & 2-DAYS PER MONTH ===
for month, clean in zip(month_cols, clean_months):
    daily_col   = f"Daily_{clean}"
    twodays_col = f"2Days_{clean}"
    
    df_merged[daily_col]   = (df_merged[month] / 30.0).round(2)
    df_merged[twodays_col] = (df_merged[daily_col] * df_merged['Qty_per_Switch'] * 2).round(2)

# ─────────────────────────────────────────────
# PART 1: TOTALS PER CHILD PART
# ─────────────────────────────────────────────
# Aggregate only the Daily columns
agg_dict = {f"Daily_{clean}": 'sum' for clean in clean_months}

totals = df_merged.groupby('Child_Part', as_index=False).agg(agg_dict)

# Add 2Days columns
for clean in clean_months:
    totals[f"2Days_{clean}"] = (totals[f"Daily_{clean}"] * 2).round(2)

# Column order
totals_cols = (
    ['Child_Part'] +
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
)

totals = totals[totals_cols]

# Save
totals_file = "Child_Totals_2Days_Per_Month.xlsx"
totals.to_excel(totals_file, index=False)
print(f"Totals file saved: {totals_file} ({len(totals)} rows)")

# ─────────────────────────────────────────────
# PART 2: DETAILED BREAKDOWN
# ─────────────────────────────────────────────
detailed_cols = (
    ['Child_Part', 'Switch_Part', 'Qty_per_Switch'] +
    month_cols +                           # original months with '
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
)

detailed = df_merged[detailed_cols]

detailed_file = "Child_Detailed_Breakdown_2Days.xlsx"
detailed.to_excel(detailed_file, index=False)
print(f"Detailed file saved: {detailed_file} ({len(detailed)} rows)")

print("Done — check both Excel files in your folder.")

KeyError: "None of [Index(['Switch_Part', 'Feb'26', 'Mar'26', 'Apr'26', 'May'26', 'Jun'26',\n       'Jul'26'],\n      dtype='object')] are in the [columns]"